# Mermaid Diagrams

`cimgraph.utils` generates [Mermaid](https://mermaid.js.org/) diagram text for CIM classes and objects:

* `get_mermaid(root)`: a class diagram of a class or a list of classes, or a mind map of an object and its neighbors.
* `get_mermaid_path(root, path)`: a diagram that follows a path of associations from a class or an object.
* `add_mermaid_path(root, path, diagram)`: adds another path to a diagram.

The functions return strings. To render one, wrap it in a ```` ```mermaid ```` fence and display it as Markdown, which JupyterLab 4.1+ and this site render:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim
from IPython.display import Markdown, display

from cimgraph import utils

def show_mermaid(diagram):
    display(Markdown(f'```mermaid\n{diagram}\n```'))

## Class Diagrams

A list of classes gives their attributes and the associations between them:

In [2]:
diagram = utils.get_mermaid([cim.ACLineSegment, cim.ACLineSegmentPhase])
print(diagram)
show_mermaid(diagram)

%%{init: {"theme":"neutral", "layout":"dagre"}}%%
classDiagram
    class ACLineSegment{
        + b0ch: float | Susceptance
        + bch: float | Susceptance
        + g0ch: float | Conductance
        + gch: float | Conductance
        + r: float | Resistance
        + r0: float | Resistance
        + shortCircuitEndTemperature: float | Temperature
        + x: float | Reactance
        + x0: float | Reactance
    }
    ACLineSegment --> "0..*" ACLineSegmentPhase : ACLineSegmentPhases 
    class ACLineSegmentPhase{
        + sequenceNumber: int
        + phase: enum:SinglePhaseKind
    }
    ACLineSegmentPhase --> "0..1" ACLineSegment : ACLineSegment 



```mermaid
%%{init: {"theme":"neutral", "layout":"dagre"}}%%
classDiagram
    class ACLineSegment{
        + b0ch: float | Susceptance
        + bch: float | Susceptance
        + g0ch: float | Conductance
        + gch: float | Conductance
        + r: float | Resistance
        + r0: float | Resistance
        + shortCircuitEndTemperature: float | Temperature
        + x: float | Reactance
        + x0: float | Reactance
    }
    ACLineSegment --> "0..*" ACLineSegmentPhase : ACLineSegmentPhases 
    class ACLineSegmentPhase{
        + sequenceNumber: int
        + phase: enum:SinglePhaseKind
    }
    ACLineSegmentPhase --> "0..1" ACLineSegment : ACLineSegment 

```

A single class also shows its parent class and all of its associations. Options:

* `show_attributes=False` hides the attributes.
* `show_inherited=True` adds the attributes and associations inherited from parent classes.
* `serialize_only=True` hides the reverse associations, which are not written to CIM XML.
* `theme` is the Mermaid theme (default `'neutral'`).

In [3]:
show_mermaid(utils.get_mermaid(cim.EnergyConsumer))

```mermaid
%%{init: {"theme":"neutral", "layout":"dagre"}}%%
classDiagram
    class EnergyConsumer{
        + customerCount: int
        + grounded: bool
        + p: float | ActivePower
        + phaseConnection: enum:PhaseShuntConnectionKind
        + q: float | ReactivePower
    }
    EnergyConnection <|-- EnergyConsumer : inherits from
    EnergyConsumer --> "0..1" House : House 
    EnergyConsumer --> "0..1" LoadResponseCharacteristic : LoadResponse 
    EnergyConsumer --> "0..1" PowerCutZone : PowerCutZone 
    EnergyConsumer --> "0..*" EnergyConsumerPhase : EnergyConsumerPhase 

```

## Object Mind Maps

For an object, `get_mermaid` draws a mind map of its attributes and the objects it references. Load the IEEE 13 bus feeder and draw the load at bus 671:

In [4]:
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(connection=XMLFile(filename='../../tests/test_models/ieee13.xml'),
                      container=feeder, distributed=False)

load = network.get_object('E26D83A0-D29D-41EF-9528-02C882FFCC0D')
show_mermaid(utils.get_mermaid(load))

```mermaid
mindmap
    E26D83A0((**EnergyConsumer**
        name: 671
        customerCount: 1
        grounded: False
        p: 1155000 watt
        q: 660000 VAr))
        [Location]
            A1B4F004((**Location**
                name: 671_Loc))
        [EquipmentContainer]
            49AD8E07((**Feeder**
                name: ieee13nodeckt))
        [BaseVoltage]
            2A158E0C((**BaseVoltage**
                name: BaseV_4.1600))
        ["Terminals"]
            F3A001BA((**Terminal**
                name: 671_T1))
        [LoadResponse]
            76011ACA((**LoadResponse**
                **Characteristic**
                name: Constant kVA))

```

## Paths

A path is a list of attribute names, or a string of them separated by dots. `attribute[index]` picks one object from a list. From an object, the path draws the objects it reaches:

In [5]:
diagram = utils.get_mermaid_path(load, 'Terminals[0].ConnectivityNode')
diagram = utils.add_mermaid_path(load, 'BaseVoltage', diagram)
show_mermaid(diagram)

```mermaid
%%{init: {"theme":"neutral"}}%%
flowchart LR
    E26D83A0("`**EnergyConsumer**
        name: 671`")
    E26D83A0 -- "Terminals[0]" --> F3A001BA
    F3A001BA("`**Terminal**
        name: 671_T1`")
    F3A001BA -- "ConnectivityNode" --> 30BE5988
    30BE5988("`**ConnectivityNode**
        name: 671`")
    E26D83A0("`**EnergyConsumer**
        name: 671`")
    E26D83A0 -- "BaseVoltage" --> 2A158E0C
    2A158E0C("`**BaseVoltage**
        name: BaseV_4.1600`")

```

From a class, the path draws the classes and associations it follows. A `__subclasses__()[index]` step goes to a subclass.

In [6]:
show_mermaid(utils.get_mermaid_path(cim.ACLineSegment, 'Terminals.ConnectivityNode',
                                    show_attributes=False))

```mermaid
%%{init: {"theme":"neutral"}}%%
classDiagram
    class ACLineSegment{
    }
    ACLineSegment --> "0..*" Terminal : Terminals 
    class Terminal{
    }
    Terminal --> "0..1" ConnectivityNode : ConnectivityNode 
    class ConnectivityNode{
    }

```